## 02. Text Preprocessing Pipeline

This notebook validates the shared cleaning, tokenization, lemmatization, and stopword removal pipeline for Cernovalis. Install the model first with `python -m spacy download en_core_web_sm`; the pipeline never downloads models automatically.

1. **Wire artifact stripping**: Remove leading `(Reuters)` wire service datelines from True articles to prevent trivial leakage.
2. **Media & tag removal**: Strip brackets/markers like `[VIDEO]`, `[WATCH]`, and HTML tags.
3. **URL & Email stripping**: Eliminate hyperlinked targets and contact addresses.
4. **Contraction expansion**: Standardize colloquialisms (`don't` -> `do not`, `it's` -> `it is`).
5. **ASCII / Unicode normalization**: Normalize accents and special glyphs.
6. **Case normalization & whitespace collapsing**: Produce consistent clean token streams.
7. **NLP preprocessing**: Tokenize with spaCy, lemmatize, and remove English stopwords while retaining negations (`no`, `not`, `never`).
8. **Dataset preparation**: Export cleaned corpus to `backend/ml/data/processed/cleaned_articles.csv`.

In [ ]:
import sys
from pathlib import Path

# Resolve project root (Cernovalis) from notebook directory
curr = Path.cwd().resolve()
for p in [curr, curr.parent, curr.parent.parent, curr.parent.parent.parent]:
    if (p / 'backend' / 'app').is_dir():
        if str(p) not in sys.path:
            sys.path.insert(0, str(p))
        break

import pandas as pd
from backend.app.services.preprocessor import TextPreprocessor

preprocessor = TextPreprocessor(use_spacy=True, keep_words=['no', 'not', 'never'])
title_preprocessor = TextPreprocessor(
    use_spacy=True, keep_words=['no', 'not', 'never'], strip_wire=False
)

## 1. Load Raw Datasets

Load `True.csv` and `Fake.csv`, assign binary labels (`1` for Real/True, `0` for Fake), and concatenate into a working DataFrame.

In [ ]:
candidates = (Path('../data/raw'), Path('backend/ml/data/raw'))
data_dir = next((p for p in candidates if p.is_dir()), None)
if data_dir is None:
    raise FileNotFoundError('Could not locate backend/ml/data/raw')

true_df = pd.read_csv(data_dir / 'True.csv')
fake_df = pd.read_csv(data_dir / 'Fake.csv')

true_df['label'] = 1
true_df['class'] = 'True'
fake_df['label'] = 0
fake_df['class'] = 'Fake'

df = pd.concat([true_df, fake_df], ignore_index=True).sample(frac=1.0, random_state=42).reset_index(drop=True)
print(f'Total combined articles: {len(df):,}')
df.head(3)

## 2. Inspect Cleaning Steps on Sample Texts

Verify that our pipeline properly handles wire datelines, HTML, URLs, and contractions.

In [ ]:
sample_texts = [
    'WASHINGTON (Reuters) - The U.S. Senate voted 51-49 on Friday to pass tax overhaul legislation.',
    'BREAKING: [VIDEO] Hillary Clinton\'s campaign couldn\'t believe this link https://t.co/abc1234 or email info@campaign.org <p>Shocking update!</p>',
    'SEATTLE/WASHINGTON (Reuters) - Amazon said it\'s opening a new facility.',
]

for i, raw in enumerate(sample_texts, 1):
    cleaned = preprocessor.clean(raw)
    print(f'Tokens: {preprocessor.preprocess_tokens(raw)}')
    print(f'Sample {i} RAW:\n  {raw}')
    print(f'Sample {i} CLEANED:\n  {cleaned}\n')

## 3. Apply Cleaning Pipeline to Dataset

We generate clean versions of both the headline (`title_clean`) and the body (`text_clean`), as well as a combined representation (`content_clean`).

In [ ]:
# Clean titles
df['title_clean'] = title_preprocessor.pipe(df['title'].fillna(''))

# Clean article text
df['text_clean'] = preprocessor.pipe(df['text'].fillna(''))

# Combined full representation
df['content_clean'] = (df['title_clean'] + ' ' + df['text_clean']).str.strip()

print('Sample cleaned records:')
df[['title', 'title_clean', 'text_clean']].head(3)

## 4. Verification and Sanity Checks

Check length distributions, verify wire service removal, and quantify empty articles.

In [ ]:
print('Reuters wire tag check in text_clean:')
reuters_count = df['text_clean'].str.contains(r'\(reuters\)|\breuters\b', regex=True).sum()
print(f'Articles with reuters in text_clean: {reuters_count:,} ({reuters_count / len(df):.2%})')

print('\nEmpty cleaned texts:')
empty_titles = (df['title_clean'] == '').sum()
empty_bodies = (df['text_clean'] == '').sum()
empty_content = (df['content_clean'] == '').sum()
print(f'Empty title_clean: {empty_titles}')
print(f'Empty text_clean: {empty_bodies}')
print(f'Empty content_clean: {empty_content}')

## 5. Export Cleaned Dataset

Save the preprocessed corpus to `backend/ml/data/processed/cleaned_articles.csv` for downstream feature engineering and modeling.

In [ ]:
output_candidates = (Path('../data/processed'), Path('backend/ml/data/processed'))
out_dir = next((p for p in output_candidates if p.is_dir()), None)
if out_dir is None:
    out_dir = Path('backend/ml/data/processed')
    out_dir.mkdir(parents=True, exist_ok=True)

export_cols = ['title', 'text', 'subject', 'date', 'label', 'class', 'title_clean', 'text_clean', 'content_clean']
out_path = out_dir / 'cleaned_articles.csv'
df[export_cols].to_csv(out_path, index=False, encoding='utf-8')
print(f'Successfully saved cleaned dataset to {out_path.resolve()} ({len(df):,} rows)')